Imports and Setup

In [ ]:
# only run this cell if modules have not been installed yet for this environment
%pip install pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --------------------------- ------------ 5.8/8.4 MB 30.5 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 21.4 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   -------- ------------------------------- 7.9/37.4 MB 36.2 MB/s eta 0:00:01
   ----------------- ---------------------- 16.5/37.4 MB 38.5 MB/s eta 0:00:01
   --------------------------- ------------ 25.4/37.4 MB 39.9 MB/s eta 0:00:01
   ------------------------------------ --- 34.3/37.4 MB 40.3 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 24.7 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ ---------------

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    make_scorer, average_precision_score, roc_auc_score, f1_score,
    recall_score, precision_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, balanced_accuracy_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 100)

Load the data

In [6]:
df = pd.read_csv('../data/diabetic_data.csv')
print(f"Data shape: {df.shape}")
df.head()

Data shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Data Audit  - Verify the shape, types and missing data

In [7]:
print("Shape of the dataset:", df.shape)
print("\nColumns types:\n", df.dtypes.value_counts())

print("\nMissing values (NaN or '?'):")
missing_values = df.isin(['?']).sum() + df.isna().sum()
print(missing_values[missing_values > 0])
print("\nMissing values percentage:")
print((missing_values[missing_values > 0] / df.shape[0]) * 100)

Shape of the dataset: (101766, 50)

Columns types:
 str      37
int64    13
Name: count, dtype: int64

Missing values (NaN or '?'):
race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64

Missing values percentage:
race                  2.233555
weight               96.858479
payer_code           39.557416
medical_specialty    49.082208
diag_1                0.020636
diag_2                0.351787
diag_3                1.398306
max_glu_serum        94.746772
A1Cresult            83.277322
dtype: float64


Target distribution

In [8]:
print("Raw target value counts:\n", df['readmitted'].value_counts())
print("\nProportion of target values:\n", df['readmitted'].value_counts(normalize=True))

# Creates binary target (1 if readmitted within 30 days, 0 otherwise)
df["readmitted_in_30_days"] = (df["readmitted"] == "<30").astype(int)
print("\nBinary target value counts:\n", df['readmitted_in_30_days'].value_counts())

print("\nProportion of binary target values:\n", df['readmitted_in_30_days'].value_counts(normalize=True))

Raw target value counts:
 readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Proportion of target values:
 readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

Binary target value counts:
 readmitted_in_30_days
0    90409
1    11357
Name: count, dtype: int64

Proportion of binary target values:
 readmitted_in_30_days
0    0.888401
1    0.111599
Name: proportion, dtype: float64
